
Stock Price Prediction with LSTM

Last Updated: July 6th, 2025

Daily Challenge: Stock Price Prediction with LSTM


👩‍🏫 👩🏿‍🏫 What You’ll learn

    How to preprocess and prepare time-series data for machine learning models.
    How to build and train an LSTM (Long Short-Term Memory) model using PyTorch.
    How to evaluate the performance of a regression model using metrics like R².


🛠️ What you will create

    A preprocessed dataset for stock price prediction.
    A trained LSTM model to predict future stock prices.


⚠️ Warning ! In this daily challenge, you need to use a VM like DigitalOcean! ⚠️


Understanding PyTorch

PyTorch is an open-source machine learning framework based on the Torch library, used for applications such as computer vision and natural language processing. It’s known for its flexibility and ease of use, making it popular for both research and production.

Key PyTorch Functions You’ll Use:

    torch.nn.Module: Base class for all neural network modules. You’ll use this to define your LSTM model.
    torch.nn.LSTM: Implements a Long Short-Term Memory (LSTM) network.
    torch.nn.Linear: Applies a linear transformation to the incoming data (i.e., a fully connected layer).
    torch.nn.Dropout: Applies dropout regularization to prevent overfitting.
    torch.optim.Adam: Implements the Adam optimization algorithm.
    torch.nn.MSELoss: Implements the Mean Squared Error loss function.
    torch.utils.data.Dataset: An abstract class representing a dataset.
    torch.utils.data.DataLoader: Combines a dataset and a sampler, and provides single- or multi-process iterators over the dataset.
    torch.Tensor: A multi-dimensional matrix containing elements of a single data type.
    torch.save and torch.load: used to save and load trained models.

For further understanding on how to use Pytorch functions, you can watch this video and this one too, good luck !


What You Need to Do

1. Install Required Libraries

Ensure you have the necessary libraries installed, including gensim, spacy, torch, and scikit-learn.

2. Load and Preprocess the Dataset

    Download the stock market dataset.
    Drop unnecessary columns and create a target column for the next day’s closing price.
    Normalize the dataset using MinMaxScaler.

3. Prepare the Dataset for Training

    Split the dataset into training, validation, and testing sets.
    Create a custom PyTorch Dataset class to handle the data.
    Use DataLoader to create iterable datasets for training and evaluation.

4. Define the LSTM Model

    Create an LSTM model using PyTorch.
    Define the model architecture, including GRU layers, dropout, and a dense layer.

5. Train the Model

    Set up the optimizer and loss function.
    Implement training and validation loops.
    Train the model for a specified number of epochs.

6. Evaluate the Model

    Calculate the R² score to evaluate the model’s performance on the test set.
    Save the scaler object for future predictions.




In [ ]:
import os
import json
import math
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# basic configuration & reproducibility
CSV_PATH = "stock_market_dataset.csv"     # <- adapt path if needed
SAVE_DIR = "artifacts_lstm_friend"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 123
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# toggles (architecture, scalers, training hyperparams)
MODEL_TYPE = "GRU"            # choices: "GRU" or "LSTM"
X_SCALER_TYPE = "robust"      # choices: "robust" or "minmax" (kept robust by default)
Y_SCALER_TYPE = "minmax"      # recommend MinMax for the target
SEQ_LEN = 32                  # sequence length (customized)
BATCH_SIZE = 64
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0
PATIENCE = 10                 # early stopping patience

# load data & harmonize column names
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Optional aliasing for robustness
if "timestamp" in df.columns and "unix" not in df.columns:
    df["unix"] = df["timestamp"]
if "volume_usdt" not in df.columns and "volumeusdt" in df.columns:
    df["volume_usdt"] = df["volumeusdt"]
if "volume_xrp" not in df.columns and "volumexpr" in df.columns:
    df["volume_xrp"] = df["volumexpr"]

required = ["open", "high", "low", "close"]
for c in required:
    if c not in df.columns:
        raise ValueError(f"Required column missing: '{c}'")

# Prefer chronological sort by unix (else try date)
if "unix" in df.columns:
    df = df.sort_values("unix").reset_index(drop=True)
elif "date" in df.columns:
    try:
        df["__parsed_date__"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.sort_values("__parsed_date__").drop(columns="__parsed_date__").reset_index(drop=True)
    except Exception:
        pass

print(df.head())
print(df.describe(include="all"))

# lightweight feature engineering (different from basic version)
df["ret_1d"] = df["close"].pct_change()
df["vola_5d"] = df["ret_1d"].rolling(5).std()
df["ema_10"] = df["close"].ewm(span=10, adjust=False).mean()
df["ema_20"] = df["close"].ewm(span=20, adjust=False).mean()
df["oc_gap"] = (df["open"] - df["close"]) / df["close"].replace(0, np.nan)
df["hl_range"] = (df["high"] - df["low"]) / df["close"].replace(0, np.nan)
if "volume_usdt" in df.columns:
    df["vol_usdt_log"] = np.log1p(df["volume_usdt"].clip(lower=0))
if "volume_xrp" in df.columns:
    df["vol_xrp_log"] = np.log1p(df["volume_xrp"].clip(lower=0))

def compute_rsi(series, window=14):
    delta = series.diff()
    up = delta.clip(lower=0)
    down = (-delta).clip(lower=0)
    roll_up = up.ewm(alpha=1/window, adjust=False).mean()
    roll_down = down.ewm(alpha=1/window, adjust=False).mean()
    rs = roll_up / (roll_down + 1e-12)
    return 100 - (100 / (1 + rs))

df["rsi_14"] = compute_rsi(df["close"], 14)

# Next-day close target
df["target"] = df["close"].shift(-1)

# Clean NaNs introduced by features/shift
df = df.replace([np.inf, -np.inf], np.nan).ffill().bfill().dropna().reset_index(drop=True)

# select feature set
feature_cols = ["close", "open", "high", "low",
                "ret_1d", "vola_5d", "ema_10", "ema_20",
                "oc_gap", "hl_range", "rsi_14"]
if "vol_usdt_log" in df.columns: feature_cols.append("vol_usdt_log")
if "vol_xrp_log" in df.columns: feature_cols.append("vol_xrp_log")
print("\nUsing features:", feature_cols)

X_raw = df[feature_cols].astype(float).values
y_raw = df["target"].astype(float).values.reshape(-1, 1)

# scalers with toggle (keep robust for X by default)
def get_scaler(name: str):
    name = name.lower()
    if name == "robust":
        return RobustScaler()
    elif name == "minmax":
        return MinMaxScaler()
    else:
        raise ValueError(f"Unknown scaler type: {name}")

scaler_X = get_scaler(X_SCALER_TYPE)
scaler_y = get_scaler(Y_SCALER_TYPE)

X_scaled = scaler_X.fit_transform(X_raw)
y_scaled = scaler_y.fit_transform(y_raw)

# build sequences (X: [T,F], y: scalar)
def make_sequences(X, y, seq_len):
    xs, ys = [], []
    for i in range(len(X) - seq_len):
        xs.append(X[i:i+seq_len, :])
        ys.append(y[i+seq_len, 0])
    return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_seq, y_seq = make_sequences(X_scaled, y_scaled, SEQ_LEN)
print("X_seq:", X_seq.shape, "y_seq:", y_seq.shape)

# chronological split (80/10/10)
n = len(X_seq)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

X_train, y_train = X_seq[:n_train], y_seq[:n_train]
X_val, y_val     = X_seq[n_train:n_train+n_val], y_seq[n_train:n_train+n_val]
X_test, y_test   = X_seq[n_train+n_val:], y_seq[n_train+n_val:]
print(f"Splits -> train:{len(X_train)} val:{len(X_val)} test:{len(X_test)}")

# PyTorch Dataset/DataLoader
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SeqDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SeqDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

# define models (GRU customized; LSTM also available)
class LSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.30):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, dropout=dropout, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(hidden_size // 2, 1),
        )
    def forward(self, x):
        out, _ = self.lstm(x)          # (B,T,H)
        h_last = out[:, -1, :]         # (B,H)
        return self.head(h_last).squeeze(-1)

class GRURegressor(nn.Module):
    def __init__(self, input_size, hidden_size=72, num_layers=2, dropout=0.25):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, dropout=dropout, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(hidden_size // 2, 1),
        )
    def forward(self, x):
        out, _ = self.gru(x)           # (B,T,H)
        h_last = out[:, -1, :]
        return self.head(h_last).squeeze(-1)

if MODEL_TYPE.upper() == "GRU":
    model = GRURegressor(input_size=X_seq.shape[2]).to(DEVICE)
else:
    model = LSTMRegressor(input_size=X_seq.shape[2]).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                       factor=0.5, patience=5, verbose=True)

# train loop with early stopping, gradient clipping, LR scheduler
def eval_loader(data_loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in data_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            loss = criterion(pred, yb)
            losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("inf")

best_val = float("inf")
best_state = None
no_improve = 0

for epoch in range(1, EPOCHS+1):
    model.train()
    train_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        train_losses.append(loss.item())

    val_loss = eval_loader(val_loader)
    scheduler.step(val_loss)
    tr_loss = float(np.mean(train_losses)) if train_losses else float("nan")
    print(f"[{epoch:03d}/{EPOCHS}] train_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | lr={optimizer.param_groups[0]['lr']:.1e}")

    if val_loss < best_val - 1e-6:
        best_val = val_loss
        best_state = model.state_dict()
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}. Best val_loss={best_val:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state)

# evaluation utilities (inverse scaling to real price space)
def predict_loader(data_loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(DEVICE)
            out = model(xb).detach().cpu().numpy().reshape(-1, 1)
            y_pred.append(out)
            y_true.append(yb.numpy().reshape(-1, 1))
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)
    true_real = scaler_y.inverse_transform(y_true).ravel()
    pred_real = scaler_y.inverse_transform(y_pred).ravel()
    return true_real, pred_real

y_true_test, y_pred_test = predict_loader(test_loader)

# baselines for comparison (real-scale)
naive_shift = np.roll(y_true_test, 1)   # persistence baseline: y_{t+1} = y_t
naive_shift[0] = y_true_test[0]

close_idx = feature_cols.index("close")
last_step_scaled = X_test[:, -1, :]
last_step_real = scaler_X.inverse_transform(last_step_scaled)
naive_last_close = last_step_real[:, close_idx]

def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mape = float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100.0)
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE%": mape, "R2": r2}

m_model = metrics(y_true_test, y_pred_test)
m_naive_shift = metrics(y_true_test, naive_shift)
m_naive_close = metrics(y_true_test, naive_last_close)

print("\n=== Test metrics (real scale) ===")
print("Model         :", m_model)
print("Naive (shift) :", m_naive_shift)
print("Naive (close) :", m_naive_close)

# quick visualization
plt.figure(figsize=(12,5))
plt.plot(y_true_test[:200], label="Actual")
plt.plot(y_pred_test[:200], label=f"Predicted ({MODEL_TYPE})", alpha=0.85)
plt.title("Actual vs Predicted (real scale) — sample")
plt.xlabel("Time steps")
plt.ylabel("Price")
plt.grid(True); plt.legend(); plt.tight_layout()
plt.show()

# save artifacts (model, scalers, predictions, config)
model_path = os.path.join(SAVE_DIR, f"{MODEL_TYPE.lower()}_best.pt")
torch.save(model.state_dict(), model_path)
joblib.dump(scaler_X, os.path.join(SAVE_DIR, "scaler_X.pkl"))
joblib.dump(scaler_y, os.path.join(SAVE_DIR, "scaler_y.pkl"))

pred_df = pd.DataFrame({
    "y_true": y_true_test,
    "y_pred": y_pred_test,
    "y_pred_naive_shift": naive_shift,
    "y_pred_naive_close": naive_last_close
})
pred_df.to_csv(os.path.join(SAVE_DIR, "predictions_test.csv"), index=False)

config = {
    "model_type": MODEL_TYPE,
    "seq_len": SEQ_LEN,
    "features": feature_cols,
    "x_scaler": X_SCALER_TYPE,
    "y_scaler": Y_SCALER_TYPE,
    "train": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "grad_clip": GRAD_CLIP,
        "scheduler": "ReduceLROnPlateau",
        "early_stopping_patience": PATIENCE
    },
    "device": str(DEVICE),
    "metrics_model": m_model,
    "metrics_naive_shift": m_naive_shift,
    "metrics_naive_close": m_naive_close
}
with open(os.path.join(SAVE_DIR, "run_config.json"), "w") as f:
    json.dump(config, f, indent=2)

print(f"\nArtifacts saved to: {SAVE_DIR}")
print(f"Best model path: {model_path}")

# helper to predict next-day close from the last SEQ_LEN rows
def predict_next_day(df_full: pd.DataFrame,
                     feature_cols: list,
                     scaler_X,
                     scaler_y,
                     seq_len: int = SEQ_LEN) -> float:
    """
    Use the last `seq_len` rows to predict the next day's close (real scale).
    Assumes df_full already contains the engineered features with the same pipeline.
    """
    X_latest = df_full[feature_cols].astype(float).values[-seq_len:]
    X_scaled_latest = scaler_X.transform(X_latest)
    x = torch.tensor(X_scaled_latest, dtype=torch.float32).unsqueeze(0).to(DEVICE)  # (1, T, F)
    model.eval()
    with torch.no_grad():
        y_scaled_pred = model(x).cpu().numpy().reshape(-1, 1)
        y_pred_real = scaler_y.inverse_transform(y_scaled_pred).ravel()[0]
    return float(y_pred_real)

next_pred = predict_next_day(df, feature_cols, scaler_X, scaler_y, SEQ_LEN)
print(f"\nNext-day close prediction (real scale): {next_pred:.6f}")